# Databricks Bulk Loader Benchmark

Benchmarks **CSV**, **Avro**, and **Parquet** Databricks bulk loaders
(`DatabricksS3CsvBulkLoader`, `DatabricksS3AvroBulkLoader`, `DatabricksS3ParquetBulkLoader`)
that stage files on S3 and ingest them via `INSERT INTO … FROM read_files(…)`.

A benchmark table is created in the configured catalog/schema **before** timing starts
and **dropped unconditionally** when the final cell exits (even on error).
Each repeat truncates the table so it starts empty.

**Timing breakdown**

| Metric | Covers |
|---|---|
| *writing* | first row sent → all local files written & last file queued for upload |
| *db_load* | writing done → `INSERT INTO` complete (final upload(s) + SQL ingestion) |
| *elapsed* | full wall-clock time (*writing* + *db_load*) |

Run cells top-to-bottom. Edit the widgets in the **Parameters** cell or pass values as Job task parameters.

Example output:

```
loader   rows/file  files  MB/file  total MB  elapsed s  stdev s  writing s  db_load s  rows/s  vs best
-------  ---------  -----  -------  --------  ---------  -------  ---------  ---------  ------  -------
csv         25,000     40      1.8      71.3      62.97     6.23      51.19      11.79  15,879  +194.7%
avro        25,000     40      2.2      86.5      52.77     1.37      27.15      25.62  18,950  +147.0%
parquet     25,000     40      2.0      79.8      21.37     0.31      16.35       5.02  46,800    +0.0%

Fastest: parquet (21.37s, 46,800 rows/s)
```

In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
from pathlib import Path

notebook_path_str = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = Path(notebook_path_str).parent
print(f"{notebook_dir=}")
project_root = Path("/Workspace") / str(notebook_dir.parent.parent)[1:]
print(f"{project_root=}")

In [0]:
import subprocess
import sys

cmd = [sys.executable, "-m", "pip", "install", "-e", f"{project_root}[avro, parquet]"]
result = subprocess.run(
    cmd,
    check=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,  # Return strings instead of byte
)
print(result.stdout)

In [0]:
%uv pip install -q "databricks-sqlalchemy>2"
%uv pip install -q "config-wrangler[aws]==1.6.0a5"

In [0]:
%restart_python

In [0]:
import functools
import gc
import logging
import os
import statistics
import time
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import Optional

import yaml
from config_wrangler.config_root import ConfigRoot
from config_wrangler.config_templates.aws.aws_session import AWS_Session
from config_wrangler.config_templates.sqlalchemy_database import SQLAlchemyDatabase
from config_wrangler.config_templates.config_hierarchy import ConfigHierarchy

from sqlalchemy import create_engine, text

from databricks.sdk import WorkspaceClient
from databricks.sdk.runtime import dbutils

from bi_etl.bulk_loaders.bulk_worker import BulkWorker
from bi_etl.bulk_loaders.databricks_s3_avro_loader import DatabricksS3AvroBulkLoader
from bi_etl.bulk_loaders.databricks_s3_csv_loader import DatabricksS3CsvBulkLoader
from bi_etl.bulk_loaders.databricks_s3_parquet_loader import DatabricksS3ParquetBulkLoader
from bi_etl.bulk_loaders.s3_bulk_load_config import S3_Bulk_Loader_Config
from bi_etl.components.row.row import Row

from tests.performance.bench_data import build_table, make_rows
from tests.performance.bench_parquet_rows_per_file import MB, print_table

logging.disable(logging.CRITICAL)


In [0]:
# ── Notebook parameters ───────────────────────────────────────────────────
# Edit the widget values in the bar above, or pass them as Databricks Job
# task parameters when running on a schedule.

dbutils.widgets.text("rows",              "1,000,000",        "Rows per run")
dbutils.widgets.text("loaders",           "csv,avro,parquet", "Loaders (comma-separated)")
dbutils.widgets.text("repeats",           "3",                "Repeats per loader")
dbutils.widgets.text("worker_chunk_size",
    str(BulkWorker.DEFAULT_CHUNK_SIZE),                       "Rows per hand-off to loader")
dbutils.widgets.text("column_multiplier", "1",                "Column multiplier (wider table)")

rows              = int(dbutils.widgets.get("rows").replace("_", "").replace(",", ""))
loaders           = [l.strip() for l in dbutils.widgets.get("loaders").split(",")]
repeats           = int(dbutils.widgets.get("repeats"))
worker_chunk_size = int(dbutils.widgets.get("worker_chunk_size"))
column_multiplier = int(dbutils.widgets.get("column_multiplier"))
config_path       = Path(dbutils.widgets.get("config_path"))

print(f"rows={rows:,}  loaders={loaders}  repeats={repeats}")
print(f"worker_chunk_size={worker_chunk_size}  column_multiplier={column_multiplier}")
print(f"config_path={config_path}")


In [0]:
# ── Configuration ─────────────────────────────────────────────────────────
# Uses pydantic BaseModel (always available in Databricks) so that
# config_wrangler’s ConfigHierarchy is not required for these classes.

class DatabricksConfig(ConfigHierarchy):
    """Connection settings for a Databricks SQL warehouse.

    Attributes
    ----------
    warehouse_name
        Display name of the SQL warehouse (HTTP path resolved via SDK).
    catalog_name
        Unity Catalog catalog that contains the target schema.
    schema_name
        Schema within *catalog* where the benchmark table is created.
    """

    warehouse_name: str
    catalog_name: str
    schema_name: str = 'default'  # plain str field, not a Spark DataFrame attribute


class BenchConfig(ConfigRoot):
    """Root configuration for the Databricks bulk-loader benchmark.

    Attributes
    ----------
    s3
        S3 bucket, folder, and authentication settings for staging files.
    databricks
        Databricks SQL warehouse connection settings.
    target_table
        Table name without the catalog/schema prefix. Fully-qualified name is
        ``<catalog>.<schema>.<target_table>``.
    """

    s3: S3_Bulk_Loader_Config
    databricks: DatabricksConfig
    target_table: str = 'bench_bulk_loader'

In [0]:
config = BenchConfig.build_config_from_yaml_env(file_name='config.yaml')

In [0]:
# ── Engine builder ─────────────────────────────────────────────────────────

def build_engine(config: DatabricksConfig):
    """Create a SQLAlchemy engine for the configured Databricks SQL warehouse.

    Looks up the warehouse HTTP path via the Databricks SDK and obtains the
    current notebook token from Databricks utilities, so no credentials need
    to be stored in the config file.

    Parameters
    ----------
    config
        Databricks connection settings.

    Returns
    -------
    sqlalchemy.engine.Engine

    Raises
    ------
    ValueError
        If no warehouse with the given name is found in this workspace.
    """
    w = WorkspaceClient()

    warehouse_defn = next(
        (wh for wh in w.warehouses.list() if wh.name == config.warehouse_name),
        None,
    )
    if warehouse_defn is None:
        raise ValueError(f"SQL warehouse '{config.warehouse_name}' not found")

    http_path = warehouse_defn.odbc_params.path
    print(f"http_path: {http_path}")

    access_token = (
        dbutils.notebook.entry_point
        .getDbutils().notebook().getContext()
        .apiToken().get()
    )

    # WorkspaceClient.config.host returns the full URL; strip the scheme
    # so the databricks+connector:// URL contains only the hostname.
    host = w.config.host.removeprefix("https://").removeprefix("http://")

    url = f'databricks://token@{host}/{config.catalog_name}'
    return create_engine(
        url,
        connect_args={
            'http_path': http_path,
            'access_token': access_token,
            'schema': config.schema_name,
        },
    )


In [0]:
# ── Table adapter ──────────────────────────────────────────────────────────

class DatabricksTableAdapter:
    """Bridge between :class:`~tests.performance.bench_data.BenchTable` and the Databricks bulk loaders.

    The loaders need ``table_object.database.bind.raw_connection()`` to run
    the ``INSERT INTO … FROM read_files(…)`` SQL.
    :class:`~tests.performance.bench_data.BenchTable` provides column metadata
    but no DB connection; this adapter supplies the missing link.

    Parameters
    ----------
    bench_table
        Column definitions and qualified table name.
    engine
        SQLAlchemy engine exposed as ``database.bind``.
    """

    def __init__(self, bench_table, engine) -> None:
        self._bench_table = bench_table
        self._engine = engine

    @property
    def qualified_table_name(self) -> str:
        """Fully-qualified three-part table name."""
        return self._bench_table.qualified_table_name

    @property
    def columns(self):
        """Column definitions in table order (SQLAlchemy BenchColumn list, not Spark)."""
        return self._bench_table.columns

    @property
    def column_names(self) -> list[str]:
        """Names of the columns in table order."""
        return self._bench_table.column_names

    @staticmethod
    def _get_python_type(sql_type) -> type:
        """Return the Python type of a SQLAlchemy type."""
        return sql_type.python_type

    @property
    def database(self):
        """Namespace with a ``bind`` attribute pointing to the SQLAlchemy engine."""
        return SimpleNamespace(bind=self._engine)


In [0]:
# ── Timed loader mixin ─────────────────────────────────────────────────────

class _TimedLoader:
    """Mixin that records timestamps around :meth:`load_from_files` and counts staged files.

    Must be listed *before* the concrete loader in the MRO.

    Attributes
    ----------
    writing_done_at
        :func:`time.perf_counter` value at :meth:`load_from_files` entry
        (all rows written, last file queued for upload).
    load_done_at
        :func:`time.perf_counter` value when :meth:`load_from_files` returns
        (S3 upload and ``INSERT INTO`` both complete).
    file_count
        Number of local files queued for upload.
    total_bytes
        Total compressed size of all queued files, in bytes.
    """

    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.writing_done_at: Optional[float] = None
        self.load_done_at:    Optional[float] = None
        self.file_count:  int = 0
        self.total_bytes: int = 0

    def _queue_file_for_upload(self, local_file) -> None:
        """Record file size then delegate to the real upload queue."""
        self.file_count  += 1
        self.total_bytes += os.path.getsize(local_file)
        super()._queue_file_for_upload(local_file)

    def load_from_files(self, local_files, table_object, *args, **kwargs) -> int:
        """Wrap the parent’s ``load_from_files`` with entry/exit timestamps."""
        self.writing_done_at = time.perf_counter()
        rows = super().load_from_files(local_files, table_object, *args, **kwargs)
        self.load_done_at = time.perf_counter()
        return rows


def _timed_loader_class(loader_class: type) -> type:
    """Return a timing-instrumented subclass of *loader_class*."""
    return type(f'Timed{loader_class.__name__}', (_TimedLoader, loader_class), {})


LOADER_CLASSES: dict[str, type] = {
    'csv':     DatabricksS3CsvBulkLoader,
    'avro':    DatabricksS3AvroBulkLoader,
    'parquet': DatabricksS3ParquetBulkLoader,
}


In [0]:
# ── Result dataclass ───────────────────────────────────────────────────────

@dataclass
class LoaderResult:
    """Timings and metrics for one benchmark run. All times are in seconds.

    Attributes
    ----------
    loader
        Name of the loader (key of :data:`LOADER_CLASSES`).
    rows_per_file
        Maximum rows per staged file, as reported by the loader.
    rows
        Number of rows sent to the loader.
    file_count
        Number of files staged to S3.
    total_bytes
        Total compressed size of all staged files, in bytes.
    elapsed
        Total time from first row sent to ``INSERT INTO`` completing.
    writing
        Time from first row sent to all files written and queued.
    db_load
        Time from writing done to end of ``INSERT INTO`` (upload + SQL).
    """

    loader:        str
    rows_per_file: int
    rows:          int
    file_count:    int
    total_bytes:   int
    elapsed:       float
    writing:       float
    db_load:       float

    @property
    def rows_per_second(self) -> float:
        """Rows per second from first row sent to ``INSERT INTO`` completing."""
        return self.rows / self.elapsed

    @property
    def mb_per_file(self) -> float:
        """Average compressed file size in MB (0.0 if no files recorded)."""
        if self.file_count == 0:
            return 0.0
        return self.total_bytes / self.file_count / MB


In [0]:
# ── DDL helpers ─────────────────────────────────────────────────────────────

def _databricks_ddl_type(sa_type) -> str:
    """Map a SQLAlchemy column type to its Databricks SQL type string.

    Parameters
    ----------
    sa_type
        A SQLAlchemy type instance.

    Returns
    -------
    str
        Databricks SQL type name, e.g. ``'BIGINT'`` or ``'DECIMAL(12, 2)'``.
    """
    from sqlalchemy import Date, DateTime, Float, Integer, Numeric
    if isinstance(sa_type, Integer):
        return 'BIGINT'
    if isinstance(sa_type, Float):
        return 'DOUBLE'
    if isinstance(sa_type, DateTime):
        return 'TIMESTAMP'
    if isinstance(sa_type, Date):
        return 'DATE'
    if isinstance(sa_type, Numeric):
        return f'DECIMAL({sa_type.precision}, {sa_type.scale})'
    return 'STRING'


def create_bench_table(engine, qualified_table_name: str, columns) -> None:
    """Create the benchmark table in Databricks if it does not already exist.

    Parameters
    ----------
    engine
        SQLAlchemy engine connected to the target Databricks warehouse.
    qualified_table_name
        Three-part table name (``catalog.schema.table``).
    columns
        Column definitions (list of :class:`~tests.performance.bench_data.BenchColumn`).
    """
    col_defs = ',\n  '.join(
        f'{col.name} {_databricks_ddl_type(col.type)}'
        for col in columns
    )
    ddl = f'CREATE TABLE IF NOT EXISTS {qualified_table_name} (\n  {col_defs}\n)'
    print(ddl)
    with engine.connect() as conn:
        conn.execute(text(ddl))


def drop_bench_table(engine, qualified_table_name: str) -> None:
    """Drop the benchmark table.

    Parameters
    ----------
    engine
        SQLAlchemy engine.
    qualified_table_name
        Three-part table name.
    """
    with engine.connect() as conn:
        conn.execute(text(f'DROP TABLE IF EXISTS {qualified_table_name}'))


def truncate_bench_table(engine, qualified_table_name: str) -> None:
    """Remove all rows from the benchmark table between runs.

    Parameters
    ----------
    engine
        SQLAlchemy engine.
    qualified_table_name
        Three-part table name.
    """
    with engine.connect() as conn:
        conn.execute(text(f'TRUNCATE TABLE {qualified_table_name}'))


In [0]:
# ── Core benchmark function ─────────────────────────────────────────────────

def run_once(
        loader_name: str,
        rows: list,
        table_adapter: DatabricksTableAdapter,
        s3_config: S3_Bulk_Loader_Config,
        worker_chunk_size: int = BulkWorker.DEFAULT_CHUNK_SIZE,
) -> LoaderResult:
    """Load *rows* once with *loader_name* into the Databricks table, fully timed.

    Parameters
    ----------
    loader_name
        Key of :data:`LOADER_CLASSES`.
    rows
        Pre-generated rows (produced outside the timed section).
    table_adapter
        Provides column schema and the database engine to the loader.
    s3_config
        S3 staging configuration (bucket, folder, credentials).
    worker_chunk_size
        Rows per hand-off from the producer to the loader thread.

    Returns
    -------
    LoaderResult

    Raises
    ------
    AssertionError
        If the row count reported by Databricks does not match rows sent.
    """
    timed_class = _timed_loader_class(LOADER_CLASSES[loader_name])
    loader = timed_class(config=s3_config)
    worker = BulkWorker(
        functools.partial(loader.load_from_iterable, table_object=table_adapter),
        chunk_size=worker_chunk_size,
    )
    gc.collect()
    worker.start()
    start = time.perf_counter()
    for row in rows:
        worker.put(row)
    worker.close_input()
    worker.join()
    if worker.exception is not None:
        raise worker.exception
    if worker.value != len(rows):
        raise AssertionError(
            f'{loader_name}: Databricks reported {worker.value} rows inserted,'
            f' sent {len(rows)}'
        )
    return LoaderResult(
        loader=loader_name,
        rows_per_file=loader.rows_per_file,
        rows=len(rows),
        file_count=loader.file_count,
        total_bytes=loader.total_bytes,
        elapsed=loader.load_done_at - start,
        writing=loader.writing_done_at - start,
        db_load=loader.load_done_at - loader.writing_done_at,
    )


In [0]:
# ── Setup ──────────────────────────────────────────────────────────────────
# Load config, connect to the warehouse, create the benchmark table, and
# pre-generate the row data.  Re-run this cell if you change parameters.
qualified_table_name = (
    f'{config.databricks.catalog_name}.'
    f'{config.databricks.schema_name}.'
    f'{config.target_table}'
)
print(f'Target table : {qualified_table_name}')

engine = build_engine(config.databricks)
bench_table = build_table(qualified_table_name, column_multiplier=column_multiplier)

# Precompute once – bench_table.columns is a plain list of BenchColumn, not a Spark attr.
bench_columns = bench_table.columns
num_columns   = len(bench_columns)
table_adapter = DatabricksTableAdapter(bench_table, engine)

print(f'Creating table {qualified_table_name}...')
create_bench_table(engine, qualified_table_name, bench_columns)

print(f'Generating {rows:,} rows ({num_columns} columns)...')
row_data = make_rows(rows, column_multiplier=column_multiplier)
print(f'{rows:,} rows ready.')


In [0]:
# ── Benchmark ─────────────────────────────────────────────────────────────
# Runs each loader *repeats* times, reports the median, then drops the table.
# The table is dropped in the finally block whether or not the benchmark succeeds.

medians: list[tuple[LoaderResult, float]] = []
try:
    print(f'{rows:,} rows of {num_columns} columns, {repeats} run(s) per loader\n')
    for loader_name in loaders:
        if loader_name not in LOADER_CLASSES:
            print(f'  {loader_name}: unknown loader – skipping')
            continue
        print(f'Testing {loader_name} with {repeats} repeat(s)...')
        runs: list[LoaderResult] = []
        for repeat in range(repeats):
            truncate_bench_table(engine, qualified_table_name)
            result = run_once(
                loader_name=loader_name,
                rows=row_data,
                table_adapter=table_adapter,
                s3_config=config.s3,
                worker_chunk_size=worker_chunk_size,
            )
            runs.append(result)
            print(f'  run {repeat + 1}/{repeats}: {result.elapsed:.2f}s', flush=True)

        median = sorted(runs, key=lambda r: r.elapsed)[len(runs) // 2]
        stdev  = statistics.stdev([r.elapsed for r in runs]) if len(runs) > 1 else 0.0
        medians.append((median, stdev))
        print(f'  {loader_name}: median {median.elapsed:.2f}s', flush=True)

    if medians:
        best = min(medians, key=lambda m: m[0].elapsed)[0]
        print()
        print_table(
            [
                'loader', 'rows/file', 'files', 'MB/file', 'total MB',
                'elapsed s', 'stdev s', 'writing s', 'db_load s', 'rows/s', 'vs best',
            ],
            [
                [
                    r.loader,
                    f'{r.rows_per_file:,}',
                    f'{r.file_count:,}',
                    f'{r.mb_per_file:.1f}',
                    f'{r.total_bytes / MB:.1f}',
                    f'{r.elapsed:.2f}',
                    f'{stdev:.2f}',
                    f'{r.writing:.2f}',
                    f'{r.db_load:.2f}',
                    f'{r.rows_per_second:,.0f}',
                    f'{r.elapsed / best.elapsed - 1:+.1%}',
                ]
                for r, stdev in medians
            ],
        )
        print(f'\nFastest: {best.loader} ({best.elapsed:.2f}s, {best.rows_per_second:,.0f} rows/s)')
finally:
    print(f'\nDropping table {qualified_table_name}...')
    drop_bench_table(engine, qualified_table_name)
    engine.dispose()
    print('Done.')
